In [1]:
from fvcore.nn import FlopCountAnalysis, flop_count_table
import os
import cv2
import time
import torch
import argparse
import numpy as np
from PIL import Image
import albumentations as A
from mmdet.apis import init_detector
from libs.datasets.pipelines import Compose
from libs.datasets.metrics.culane_metric import interp
from libs.utils.visualizer import visualize_lanes
from tqdm import tqdm
import shutil
import natsort


def load_models(model_path, config_path, device):
    print(f"Model_path: {model_path}")
    print(f"Config_path: {config_path}")
    print(f"Device: {device}")
    
    model = init_detector(config_path, model_path, device=device)
    model.bbox_head.test_cfg.as_lanes = False
    
    return model

model_path = '/work/CLRerNet/work_dirs/clrernet_culane_segman/run1/epoch_15.pth'
config_path = '/work/CLRerNet/work_dirs/clrernet_culane_segman/run1/20250717_160046/vis_data/config.py'
device='cuda:0'

model = load_models(model_path, config_path, device)
input_img = torch.ones(1, 3, 320, 800)
input_img = input_img.to('cuda:0')

flops = FlopCountAnalysis(model, input_img)

print("flops: ", flops.total())
print(flop_count_table(flops))


/home/docker/.pyenv/versions/3.10.13/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model_path: /work/CLRerNet/work_dirs/clrernet_culane_segman/run1/epoch_15.pth
Config_path: /work/CLRerNet/work_dirs/clrernet_culane_segman/run1/20250717_160046/vis_data/config.py
Device: cuda:0
Loads checkpoint by local backend from path: /work/CLRerNet/work_dirs/clrernet_culane_segman/run1/epoch_15.pth
34680 data are loaded


/home/docker/.pyenv/versions/3.10.13/lib/python3.10/site-packages/mmdet/apis/inference.py:108: UserWarning: palette does not exist, random is used by default. You can also set the palette to customize.
  warnings.warn(
Unsupported operator aten::gelu encountered 13 time(s)
Unsupported operator aten::mul encountered 142 time(s)
Unsupported operator aten::sin encountered 8 time(s)
Unsupported operator aten::repeat encountered 37 time(s)
Unsupported operator aten::cos encountered 8 time(s)
Unsupported operator aten::add encountered 100 time(s)
Unsupported operator aten::neg encountered 38 time(s)
Unsupported operator prim::PythonOp.NeighborhoodAttention2DQKAutogradFunction encountered 10 time(s)
Unsupported operator aten::softmax encountered 15 time(s)
Unsupported operator prim::PythonOp.NeighborhoodAttention2DAVAutogradFunction encountered 10 time(s)
Unsupported operator aten::silu encountered 8 time(s)
Unsupported operator prim::PythonOp.CrossScanTriton encountered 8 time(s)
Unsupported

flops:  5473212928.0
| module                                                | #parameters or shape   | #flops     |
|:------------------------------------------------------|:-----------------------|:-----------|
| model                                                 | 2.847M                 | 5.473G     |
|  backbone                                             |  2.281M                |  2.786G    |
|   backbone.patch_embed                                |   16.752K              |   0.402G   |
|    backbone.patch_embed.0                             |    0.432K              |    27.648M |
|    backbone.patch_embed.1                             |    32                  |    2.048M  |
|    backbone.patch_embed.3                             |    2.304K              |    0.147G  |
|    backbone.patch_embed.4                             |    32                  |    2.048M  |
|    backbone.patch_embed.6                             |    4.608K              |    73.728M |
|    backbone.patch

In [9]:
def display_flops(model_path, config_path, device, input_shape):

    model = load_models(model_path, config_path, device)
    input_img = torch.ones(input_shape[0], input_shape[1], input_shape[2], input_shape[3])
    input_img = input_img.to(device)

    flops = FlopCountAnalysis(model, input_img)

    print("total flops: ", flops.total())
    print(flop_count_table(flops))

#model_path = '/work/CLRerNet/work_dirs/clrernet_culane_segman/run1/epoch_15.pth'
#config_path = '/work/CLRerNet/work_dirs/clrernet_culane_segman/run1/20250717_160046/vis_data/config.py'
device='cuda:0'

model_path = '/work/CLRerNet/work_dirs/clrernet_culane_segman/run3/epoch_15.pth'
config_path = '/work/CLRerNet/work_dirs/clrernet_culane_segman/run3/20250912_154813/vis_data/config.py'

display_flops(model_path, config_path, device, (1, 3, 320, 800))

Model_path: /work/CLRerNet/work_dirs/clrernet_culane_segman/run3/epoch_15.pth
Config_path: /work/CLRerNet/work_dirs/clrernet_culane_segman/run3/20250912_154813/vis_data/config.py
Device: cuda:0
Loads checkpoint by local backend from path: /work/CLRerNet/work_dirs/clrernet_culane_segman/run3/epoch_15.pth
34680 data are loaded


/home/docker/.pyenv/versions/3.10.13/lib/python3.10/site-packages/mmdet/apis/inference.py:108: UserWarning: palette does not exist, random is used by default. You can also set the palette to customize.
  warnings.warn(
Unsupported operator aten::gelu encountered 21 time(s)
Unsupported operator aten::mul encountered 218 time(s)
Unsupported operator aten::sin encountered 8 time(s)
Unsupported operator aten::repeat encountered 37 time(s)
Unsupported operator aten::cos encountered 8 time(s)
Unsupported operator aten::add encountered 172 time(s)
Unsupported operator aten::neg encountered 68 time(s)
Unsupported operator prim::PythonOp.NeighborhoodAttention2DQKAutogradFunction encountered 18 time(s)
Unsupported operator aten::softmax encountered 25 time(s)
Unsupported operator prim::PythonOp.NeighborhoodAttention2DAVAutogradFunction encountered 18 time(s)
Unsupported operator aten::silu encountered 14 time(s)
Unsupported operator prim::PythonOp.CrossScanTriton encountered 14 time(s)
Unsupport

total flops:  26515807248.0
| module                                                | #parameters or shape   | #flops     |
|:------------------------------------------------------|:-----------------------|:-----------|
| model                                                 | 26.1M                  | 26.516G    |
|  backbone                                             |  23.955M               |  19.465G   |
|   backbone.patch_embed                                |   65.76K               |   1.542G   |
|    backbone.patch_embed.0                             |    0.864K              |    55.296M |
|    backbone.patch_embed.1                             |    64                  |    4.096M  |
|    backbone.patch_embed.3                             |    9.216K              |    0.59G   |
|    backbone.patch_embed.4                             |    64                  |    4.096M  |
|    backbone.patch_embed.6                             |    18.432K             |    0.295G  |
|    backbon

In [3]:
device="cuda:0"
model_path = "/work/CLRerNet/work_dirs/clrernet_culane_dla34/run1/epoch_15.pth"
config_path = "/work/CLRerNet/work_dirs/clrernet_culane_dla34/run1/20250717_022217/vis_data/config.py"

display_flops(model_path, config_path, device, (1, 3, 320, 800))

Model_path: /work/CLRerNet/work_dirs/clrernet_culane_dla34/run1/epoch_15.pth
Config_path: /work/CLRerNet/work_dirs/clrernet_culane_dla34/run1/20250717_022217/vis_data/config.py
Device: cuda:0
Loads checkpoint by local backend from path: /work/CLRerNet/work_dirs/clrernet_culane_dla34/run1/epoch_15.pth
34680 data are loaded


/home/docker/.pyenv/versions/3.10.13/lib/python3.10/site-packages/mmdet/apis/inference.py:108: UserWarning: palette does not exist, random is used by default. You can also set the palette to customize.
  warnings.warn(
Unsupported operator aten::max_pool2d encountered 6 time(s)
Unsupported operator aten::add_ encountered 17 time(s)
Unsupported operator aten::clone encountered 23 time(s)
Unsupported operator aten::repeat encountered 21 time(s)
Unsupported operator aten::mul encountered 36 time(s)
Unsupported operator aten::add encountered 10 time(s)
Unsupported operator aten::tan encountered 4 time(s)
Unsupported operator aten::sub encountered 14 time(s)
Unsupported operator aten::div encountered 4 time(s)
Unsupported operator aten::div_ encountered 4 time(s)
Unsupported operator aten::softmax encountered 3 time(s)
The following submodules of the model were never called during the trace of the graph. They may be unused, or they were accessed by direct calls to .forward() or via other py

total flops:  18377148928
| module                                                | #parameters or shape   | #flops     |
|:------------------------------------------------------|:-----------------------|:-----------|
| model                                                 | 15.827M                | 18.377G    |
|  backbone.model                                       |  15.229M               |  15.661G   |
|   backbone.model.base_layer                           |   2.384K               |   0.61G    |
|    backbone.model.base_layer.0                        |    2.352K              |    0.602G  |
|    backbone.model.base_layer.1                        |    32                  |    8.192M  |
|   backbone.model.level0                               |   2.336K               |   0.598G   |
|    backbone.model.level0.0                            |    2.304K              |    0.59G   |
|    backbone.model.level0.1                            |    32                  |    8.192M  |
|   backbone.m

In [4]:
device="cuda:0"
model_path = "/work/CLRerNet/work_dirs/clrernet_culane_segman_ssmfpn/run9/epoch_15.pth"
config_path = "/work/CLRerNet/work_dirs/clrernet_culane_segman_ssmfpn/run9/20250918_183649/vis_data/config.py"

display_flops(model_path, config_path, device, (1, 3, 320, 800))

Model_path: /work/CLRerNet/work_dirs/clrernet_culane_segman_ssmfpn/run9/epoch_15.pth
Config_path: /work/CLRerNet/work_dirs/clrernet_culane_segman_ssmfpn/run9/20250918_183649/vis_data/config.py
Device: cuda:0
Loads checkpoint by local backend from path: /work/CLRerNet/work_dirs/clrernet_culane_segman_ssmfpn/run9/epoch_15.pth
34680 data are loaded


/home/docker/.pyenv/versions/3.10.13/lib/python3.10/site-packages/mmdet/apis/inference.py:108: UserWarning: palette does not exist, random is used by default. You can also set the palette to customize.
  warnings.warn(
Unsupported operator aten::gelu encountered 21 time(s)
Unsupported operator aten::mul encountered 263 time(s)
Unsupported operator aten::sin encountered 20 time(s)
Unsupported operator aten::repeat encountered 61 time(s)
Unsupported operator aten::cos encountered 20 time(s)
Unsupported operator aten::add encountered 187 time(s)
Unsupported operator aten::neg encountered 80 time(s)
Unsupported operator prim::PythonOp.NeighborhoodAttention2DQKAutogradFunction encountered 21 time(s)
Unsupported operator aten::softmax encountered 28 time(s)
Unsupported operator prim::PythonOp.NeighborhoodAttention2DAVAutogradFunction encountered 21 time(s)
Unsupported operator aten::silu encountered 17 time(s)
Unsupported operator prim::PythonOp.CrossScanTriton encountered 17 time(s)
Unsuppo

total flops:  26134702224.0
| module                                                | #parameters or shape   | #flops     |
|:------------------------------------------------------|:-----------------------|:-----------|
| model                                                 | 26.054M                | 26.135G    |
|  backbone                                             |  23.955M               |  19.465G   |
|   backbone.patch_embed                                |   65.76K               |   1.542G   |
|    backbone.patch_embed.0                             |    0.864K              |    55.296M |
|    backbone.patch_embed.1                             |    64                  |    4.096M  |
|    backbone.patch_embed.3                             |    9.216K              |    0.59G   |
|    backbone.patch_embed.4                             |    64                  |    4.096M  |
|    backbone.patch_embed.6                             |    18.432K             |    0.295G  |
|    backbon

In [5]:
device="cuda:0"
model_path = "/work/CLRerNet/work_dirs/clrernet_culane_segman_rcformer/run2/epoch_15.pth"
config_path = "/work/CLRerNet/work_dirs/clrernet_culane_segman_rcformer/run2/20250918_184406/vis_data/config.py"

display_flops(model_path, config_path, device, (1, 3, 320, 800))

Model_path: /work/CLRerNet/work_dirs/clrernet_culane_segman_rcformer/run2/epoch_15.pth
Config_path: /work/CLRerNet/work_dirs/clrernet_culane_segman_rcformer/run2/20250918_184406/vis_data/config.py
Device: cuda:0
Loads checkpoint by local backend from path: /work/CLRerNet/work_dirs/clrernet_culane_segman_rcformer/run2/epoch_15.pth
34680 data are loaded


/home/docker/.pyenv/versions/3.10.13/lib/python3.10/site-packages/mmdet/apis/inference.py:108: UserWarning: palette does not exist, random is used by default. You can also set the palette to customize.
  warnings.warn(
Unsupported operator aten::gelu encountered 39 time(s)
Unsupported operator aten::mul encountered 236 time(s)
Unsupported operator aten::sin encountered 8 time(s)
Unsupported operator aten::repeat encountered 37 time(s)
Unsupported operator aten::cos encountered 8 time(s)
Unsupported operator aten::add encountered 226 time(s)
Unsupported operator aten::neg encountered 68 time(s)
Unsupported operator prim::PythonOp.NeighborhoodAttention2DQKAutogradFunction encountered 18 time(s)
Unsupported operator aten::softmax encountered 43 time(s)
Unsupported operator prim::PythonOp.NeighborhoodAttention2DAVAutogradFunction encountered 18 time(s)
Unsupported operator aten::silu encountered 14 time(s)
Unsupported operator prim::PythonOp.CrossScanTriton encountered 14 time(s)
Unsupport

total flops:  31020055248.0
| module                                                | #parameters or shape   | #flops     |
|:------------------------------------------------------|:-----------------------|:-----------|
| model                                                 | 61.775M                | 31.02G     |
|  backbone                                             |  23.955M               |  19.465G   |
|   backbone.patch_embed                                |   65.76K               |   1.542G   |
|    backbone.patch_embed.0                             |    0.864K              |    55.296M |
|    backbone.patch_embed.1                             |    64                  |    4.096M  |
|    backbone.patch_embed.3                             |    9.216K              |    0.59G   |
|    backbone.patch_embed.4                             |    64                  |    4.096M  |
|    backbone.patch_embed.6                             |    18.432K             |    0.295G  |
|    backbon

In [6]:
device="cuda:0"
model_path = "/work/CLRerNet/work_dirs/clrernet_culane_segman_decneck/run2/epoch_15.pth"
config_path = "/work/CLRerNet/work_dirs/clrernet_culane_segman_decneck/run2/20250918_184523/vis_data/config.py"

display_flops(model_path, config_path, device, (1, 3, 320, 800))

Model_path: /work/CLRerNet/work_dirs/clrernet_culane_segman_decneck/run2/epoch_15.pth
Config_path: /work/CLRerNet/work_dirs/clrernet_culane_segman_decneck/run2/20250918_184523/vis_data/config.py
Device: cuda:0
Loads checkpoint by local backend from path: /work/CLRerNet/work_dirs/clrernet_culane_segman_decneck/run2/epoch_15.pth
34680 data are loaded


/home/docker/.pyenv/versions/3.10.13/lib/python3.10/site-packages/mmdet/apis/inference.py:108: UserWarning: palette does not exist, random is used by default. You can also set the palette to customize.
  warnings.warn(
Unsupported operator aten::gelu encountered 28 time(s)
Unsupported operator aten::mul encountered 227 time(s)
Unsupported operator aten::sin encountered 8 time(s)
Unsupported operator aten::repeat encountered 37 time(s)
Unsupported operator aten::cos encountered 8 time(s)
Unsupported operator aten::add encountered 203 time(s)
Unsupported operator aten::neg encountered 70 time(s)
Unsupported operator prim::PythonOp.NeighborhoodAttention2DQKAutogradFunction encountered 18 time(s)
Unsupported operator aten::softmax encountered 30 time(s)
Unsupported operator prim::PythonOp.NeighborhoodAttention2DAVAutogradFunction encountered 18 time(s)
Unsupported operator aten::silu encountered 14 time(s)
Unsupported operator prim::PythonOp.CrossScanTriton encountered 16 time(s)
Unsupport

total flops:  32191351248.0
| module                                                | #parameters or shape   | #flops     |
|:------------------------------------------------------|:-----------------------|:-----------|
| model                                                 | 48.7M                  | 32.191G    |
|  backbone                                             |  23.955M               |  19.465G   |
|   backbone.patch_embed                                |   65.76K               |   1.542G   |
|    backbone.patch_embed.0                             |    0.864K              |    55.296M |
|    backbone.patch_embed.1                             |    64                  |    4.096M  |
|    backbone.patch_embed.3                             |    9.216K              |    0.59G   |
|    backbone.patch_embed.4                             |    64                  |    4.096M  |
|    backbone.patch_embed.6                             |    18.432K             |    0.295G  |
|    backbon

In [7]:
device="cuda:0"
model_path = "/work/CLRerNet/work_dirs/clrernet_culane_segman_encdec/run3/epoch_15.pth"
config_path = "/work/CLRerNet/work_dirs/clrernet_culane_segman_encdec/run3/20250915_152309/vis_data/config.py"

display_flops(model_path, config_path, device, (1, 3, 320, 800))

Model_path: /work/CLRerNet/work_dirs/clrernet_culane_segman_encdec/run3/epoch_15.pth
Config_path: /work/CLRerNet/work_dirs/clrernet_culane_segman_encdec/run3/20250915_152309/vis_data/config.py
Device: cuda:0
Loads checkpoint by local backend from path: /work/CLRerNet/work_dirs/clrernet_culane_segman_encdec/run3/epoch_15.pth
34680 data are loaded


/home/docker/.pyenv/versions/3.10.13/lib/python3.10/site-packages/mmdet/apis/inference.py:108: UserWarning: palette does not exist, random is used by default. You can also set the palette to customize.
  warnings.warn(
Unsupported operator aten::gelu encountered 21 time(s)
Unsupported operator aten::mul encountered 218 time(s)
Unsupported operator aten::sin encountered 8 time(s)
Unsupported operator aten::repeat encountered 37 time(s)
Unsupported operator aten::cos encountered 8 time(s)
Unsupported operator aten::add encountered 172 time(s)
Unsupported operator aten::neg encountered 68 time(s)
Unsupported operator prim::PythonOp.NeighborhoodAttention2DQKAutogradFunction encountered 18 time(s)
Unsupported operator aten::softmax encountered 25 time(s)
Unsupported operator prim::PythonOp.NeighborhoodAttention2DAVAutogradFunction encountered 18 time(s)
Unsupported operator aten::silu encountered 14 time(s)
Unsupported operator prim::PythonOp.CrossScanTriton encountered 14 time(s)
Unsupport

total flops:  26515807248.0
| module                                                | #parameters or shape   | #flops     |
|:------------------------------------------------------|:-----------------------|:-----------|
| model                                                 | 30.201M                | 26.516G    |
|  backbone                                             |  23.955M               |  19.465G   |
|   backbone.patch_embed                                |   65.76K               |   1.542G   |
|    backbone.patch_embed.0                             |    0.864K              |    55.296M |
|    backbone.patch_embed.1                             |    64                  |    4.096M  |
|    backbone.patch_embed.3                             |    9.216K              |    0.59G   |
|    backbone.patch_embed.4                             |    64                  |    4.096M  |
|    backbone.patch_embed.6                             |    18.432K             |    0.295G  |
|    backbon

In [8]:
device="cuda:0"
model_path = "/work/CLRerNet/work_dirs/clrernet_culane_segman_encdec_ssmfpn/run8/epoch_15.pth"
config_path = "/work/CLRerNet/work_dirs/clrernet_culane_segman_encdec_ssmfpn/run8/20250918_185010/vis_data/config.py"

display_flops(model_path, config_path, device, (1, 3, 320, 800))

Model_path: /work/CLRerNet/work_dirs/clrernet_culane_segman_encdec_ssmfpn/run8/epoch_15.pth
Config_path: /work/CLRerNet/work_dirs/clrernet_culane_segman_encdec_ssmfpn/run8/20250918_185010/vis_data/config.py
Device: cuda:0
Loads checkpoint by local backend from path: /work/CLRerNet/work_dirs/clrernet_culane_segman_encdec_ssmfpn/run8/epoch_15.pth
34680 data are loaded


/home/docker/.pyenv/versions/3.10.13/lib/python3.10/site-packages/mmdet/apis/inference.py:108: UserWarning: palette does not exist, random is used by default. You can also set the palette to customize.
  warnings.warn(
Unsupported operator aten::gelu encountered 21 time(s)
Unsupported operator aten::mul encountered 263 time(s)
Unsupported operator aten::sin encountered 20 time(s)
Unsupported operator aten::repeat encountered 61 time(s)
Unsupported operator aten::cos encountered 20 time(s)
Unsupported operator aten::add encountered 187 time(s)
Unsupported operator aten::neg encountered 80 time(s)
Unsupported operator prim::PythonOp.NeighborhoodAttention2DQKAutogradFunction encountered 21 time(s)
Unsupported operator aten::softmax encountered 28 time(s)
Unsupported operator prim::PythonOp.NeighborhoodAttention2DAVAutogradFunction encountered 21 time(s)
Unsupported operator aten::silu encountered 17 time(s)
Unsupported operator prim::PythonOp.CrossScanTriton encountered 17 time(s)
Unsuppo

total flops:  26134702224.0
| module                                                | #parameters or shape   | #flops     |
|:------------------------------------------------------|:-----------------------|:-----------|
| model                                                 | 30.146M                | 26.135G    |
|  backbone                                             |  23.955M               |  19.465G   |
|   backbone.patch_embed                                |   65.76K               |   1.542G   |
|    backbone.patch_embed.0                             |    0.864K              |    55.296M |
|    backbone.patch_embed.1                             |    64                  |    4.096M  |
|    backbone.patch_embed.3                             |    9.216K              |    0.59G   |
|    backbone.patch_embed.4                             |    64                  |    4.096M  |
|    backbone.patch_embed.6                             |    18.432K             |    0.295G  |
|    backbon

In [10]:
device="cuda:0"
model_path = "/work/CLRerNet/work_dirs/clrernet_culane_segman_encdec_rcformer/run5/epoch_15.pth"
config_path = "/work/CLRerNet/work_dirs/clrernet_culane_segman_encdec_rcformer/run5/20250918_184118/vis_data/config.py"

display_flops(model_path, config_path, device, (1, 3, 320, 800))

Model_path: /work/CLRerNet/work_dirs/clrernet_culane_segman_encdec_rcformer/run5/epoch_15.pth
Config_path: /work/CLRerNet/work_dirs/clrernet_culane_segman_encdec_rcformer/run5/20250918_184118/vis_data/config.py
Device: cuda:0
Loads checkpoint by local backend from path: /work/CLRerNet/work_dirs/clrernet_culane_segman_encdec_rcformer/run5/epoch_15.pth
34680 data are loaded


/home/docker/.pyenv/versions/3.10.13/lib/python3.10/site-packages/mmdet/apis/inference.py:108: UserWarning: palette does not exist, random is used by default. You can also set the palette to customize.
  warnings.warn(
Unsupported operator aten::gelu encountered 39 time(s)
Unsupported operator aten::mul encountered 236 time(s)
Unsupported operator aten::sin encountered 8 time(s)
Unsupported operator aten::repeat encountered 37 time(s)
Unsupported operator aten::cos encountered 8 time(s)
Unsupported operator aten::add encountered 226 time(s)
Unsupported operator aten::neg encountered 68 time(s)
Unsupported operator prim::PythonOp.NeighborhoodAttention2DQKAutogradFunction encountered 18 time(s)
Unsupported operator aten::softmax encountered 43 time(s)
Unsupported operator prim::PythonOp.NeighborhoodAttention2DAVAutogradFunction encountered 18 time(s)
Unsupported operator aten::silu encountered 14 time(s)
Unsupported operator prim::PythonOp.CrossScanTriton encountered 14 time(s)
Unsupport

total flops:  31020055248.0
| module                                                | #parameters or shape   | #flops     |
|:------------------------------------------------------|:-----------------------|:-----------|
| model                                                 | 65.876M                | 31.02G     |
|  backbone                                             |  23.955M               |  19.465G   |
|   backbone.patch_embed                                |   65.76K               |   1.542G   |
|    backbone.patch_embed.0                             |    0.864K              |    55.296M |
|    backbone.patch_embed.1                             |    64                  |    4.096M  |
|    backbone.patch_embed.3                             |    9.216K              |    0.59G   |
|    backbone.patch_embed.4                             |    64                  |    4.096M  |
|    backbone.patch_embed.6                             |    18.432K             |    0.295G  |
|    backbon

In [11]:
device="cuda:0"
model_path = "/work/CLRerNet/work_dirs/clrernet_culane_segman_encdec_decneck/run10/epoch_15.pth"
config_path = "/work/CLRerNet/work_dirs/clrernet_culane_segman_encdec_decneck/run10/20250918_184800/vis_data/config.py"

display_flops(model_path, config_path, device, (1, 3, 320, 800))

Model_path: /work/CLRerNet/work_dirs/clrernet_culane_segman_encdec_decneck/run10/epoch_15.pth
Config_path: /work/CLRerNet/work_dirs/clrernet_culane_segman_encdec_decneck/run10/20250918_184800/vis_data/config.py
Device: cuda:0
Loads checkpoint by local backend from path: /work/CLRerNet/work_dirs/clrernet_culane_segman_encdec_decneck/run10/epoch_15.pth
34680 data are loaded


/home/docker/.pyenv/versions/3.10.13/lib/python3.10/site-packages/mmdet/apis/inference.py:108: UserWarning: palette does not exist, random is used by default. You can also set the palette to customize.
  warnings.warn(
Unsupported operator aten::gelu encountered 28 time(s)
Unsupported operator aten::mul encountered 227 time(s)
Unsupported operator aten::sin encountered 8 time(s)
Unsupported operator aten::repeat encountered 37 time(s)
Unsupported operator aten::cos encountered 8 time(s)
Unsupported operator aten::add encountered 203 time(s)
Unsupported operator aten::neg encountered 70 time(s)
Unsupported operator prim::PythonOp.NeighborhoodAttention2DQKAutogradFunction encountered 18 time(s)
Unsupported operator aten::softmax encountered 30 time(s)
Unsupported operator prim::PythonOp.NeighborhoodAttention2DAVAutogradFunction encountered 18 time(s)
Unsupported operator aten::silu encountered 14 time(s)
Unsupported operator prim::PythonOp.CrossScanTriton encountered 16 time(s)
Unsupport

total flops:  32191351248.0
| module                                                | #parameters or shape   | #flops     |
|:------------------------------------------------------|:-----------------------|:-----------|
| model                                                 | 52.801M                | 32.191G    |
|  backbone                                             |  23.955M               |  19.465G   |
|   backbone.patch_embed                                |   65.76K               |   1.542G   |
|    backbone.patch_embed.0                             |    0.864K              |    55.296M |
|    backbone.patch_embed.1                             |    64                  |    4.096M  |
|    backbone.patch_embed.3                             |    9.216K              |    0.59G   |
|    backbone.patch_embed.4                             |    64                  |    4.096M  |
|    backbone.patch_embed.6                             |    18.432K             |    0.295G  |
|    backbon